In [159]:
import pandas as pd
df =pd.read_excel("./1.xlsx")

In [160]:
df['商品售后'] = df['商品售后'].replace('', '无').fillna('无')

In [161]:
#keywords=['10001483659828','10001335603671','10001388270034','10001171481605']
# 10000229167469,10000229407532,10000265046381,10000270409428,10000269553640,10000273809516
#df['商品编码(平台)'].isin(keywords)
#df['商品编码(平台)'] = df['商品编码(平台)'].astype(str)


#删除其他行
#df.drop(index= df[ df['商品编码(平台)'].isin(keywords)==False].index,inplace=True)
#df.reset_index(drop=True,inplace= True)

#influencer_id = ['洋奢塑身美体臻选店','云锦柔坊甄选','洋奢思盈服饰内衣','舒简内衣甄选','拾光衣舍服饰店', '润禾舒适优品','洋奢女装内衣服饰','洋奢塑形服饰内衣','洋奢上善臻选美体内衣服饰','洋奢内衣好物精选', '-']
#other_influencers = df[~df['带货账号昵称'].isin(influencer_id)]
#print(f"\n其他达人信息:{other_influencers[['带货账号昵称']].drop_duplicates()}")
#df = df[df['带货账号昵称'].isin(influencer_id)]

In [162]:
df["是否刷单"] = df["商家备注"].str.contains("物品", na=False)

In [163]:
df = df[~df["是否刷单"]].copy()

In [164]:
df.rename(columns={"订单下单时间":"下单时间"},inplace=True)

In [165]:
df

,订单号,下单时间,订单发货时间,订单确认收货时间,订单完成结算时间,订单状态,发货方式,收件人姓名,收件人地址,省,...,商品数量,商品平台券优惠,商品平均运费,活动商家补贴,定制信息,定制预览图,商品发货,商品售后,商品已退款金额,是否刷单
0,3740332639623747840,2026-11-05 10:48:08,-,-,-,待发货,快递,杨**,广东省茂名市茂南区****,广东省,...,1,0.0,NaN,0.0,NaN,NaN,未发货,无,0.0,False
1,3740332599229207552,2026-11-05 10:46:07,-,-,-,待付款,快递,李**,广东省佛山市三水区****,广东省,...,1,0.0,NaN,0.0,NaN,NaN,未发货,无,0.0,False
2,3740332593225308416,2026-11-05 10:45:20,-,-,-,待发货,快递,紫**,广东省广州市增城区****,广东省,...,1,0.0,NaN,0.0,NaN,NaN,未发货,无,0.0,False
3,3740332542744251136,2026-11-05 10:42:26,-,-,-,待付款,快递,李**,广东省佛山市三水区****,广东省,...,1,0.0,NaN,0.0,NaN,NaN,未发货,无,0.0,False
4,3740332415829068288,2026-11-05 10:33:55,-,-,-,已取消,快递,刘**,广东省汕头市潮南区****,广东省,...,1,0.0,NaN,0.0,NaN,NaN,未发货,退款完成,59.9,False
5,3740332373416785408,2026-11-05 10:31:13,2026-11-05 10:39:08,-,-,已发货,快递,李**,广东省湛江市麻章区****,广东省,...,1,0.0,NaN,0.0,NaN,NaN,已发货,无,0.0,False
6,3740332349100311552,2026-11-05 10:29:41,2026-11-05 10:39:08,-,-,已发货,快递,何*,广西壮族自治区防城港市港口区****,广西壮族自治区,...,1,0.0,NaN,0.0,NaN,NaN,已发货,无,0.0,False
7,3740332341474246656,2026-11-05 10:29:11,2026-11-05 10:39:08,-,-,已发货,快递,钟**,广东省中山市石岐街道****,广东省,...,1,0.0,NaN,0.0,NaN,NaN,已发货,无,0.0,False
8,3740332254787449344,2026-11-05 10:23:46,2026-11-05 10:39:08,-,-,已发货,快递,池**,浙江省台州市临海市****,浙江省,...,1,0.0,NaN,0.0,NaN,NaN,已发货,无,0.0,False
9,3740332113171757312,2026-11-05 10:14:45,2026-11-05 10:23:58,-,-,已发货,快递,王**,福建省福州市长乐区****,福建省,...,1,0.0,NaN,0.0,NaN,NaN,已发货,无,0.0,False


In [166]:
if '\t' in df.loc[0,'下单时间']:
    df['下单时间'] = df['下单时间'].str[1:]

In [167]:
from datetime import datetime
df["下单时间"] = pd.to_datetime(df["下单时间"],format = "%Y-%m-%d %H:%M:%S")
df["下单时间"] = df["下单时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df["销售金额"]= (df['商品数量'] * df['商品价格(单件)']) + (df['订单运费'])
df1 = pd.read_excel("商品编码.xlsx")
df1.rename(columns={"商品名":"产品名称","商家编码":"SKU编码(自定义)"},inplace=True)
df1 = df1[["SKU编码(自定义)","产品名称"]]
df= pd.merge(df,df1,on="SKU编码(自定义)",how="left")

In [168]:
# 缺码表
lackCode = df[df['产品名称'].isnull()]['SKU编码(自定义)'].value_counts()
lackCode

Series([], Name: SKU编码(自定义), dtype: int64)

In [169]:
lackCode.to_excel('缺码表.xlsx')

In [170]:
df['订单数'] = df['商品数量']

In [171]:
from datetime import datetime
#df["支付时间"] = df["支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["支付时间"] = pd.to_datetime(df["支付时间"],format = "%Y-%m-%d %H:%M:%S",errors='coerce')
df["支付时间"] = df["支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["支付时间"] = df["支付时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [172]:
df["下单单量"] = df['订单数']
df["支付单量"] = 0
df["发货单量"] = 0
df["发货前退款单量"] = 0

In [173]:
for i in range(0,len(df)):
    if df.loc[i,"支付时间"] != "1970-01-01":
        df.loc[i,"支付单量"] = df.loc[i,'订单数']
        df.loc[i,"支付金额"] = df.loc[i,'订单实际收款金额']

In [174]:
for i in range(0,len(df)):
    if df.loc[i,"商品发货"] == "已发货" :
        df.loc[i,"发货单量"] = df.loc[i,'订单数']

In [175]:
for i in range(0,len(df)):
    if df.loc[i,"支付时间"] != "1970-01-01" and  df.loc[i,"商品发货"] == "未发货" and df.loc[i,"商品售后"] == "退款完成":
        df.loc[i,"发货前退款单量"] = df.loc[i,'订单数']

In [176]:
df["已签收"] = 0
df["发货中1"] = 0
df["售后中"] = 0
df["已退款"] = 0
df["预计签收单量"] = 0
df["发货中2"] = 0
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已完成" and ((df.loc[i,'商品售后'] == "无") or (df.loc[i,'商品售后'] == "用户取消申请") or (df.loc[i,'商品售后'] == "平台处理完成，售后关闭") or (df.loc[i,'商品售后'] == "NaN")):
         df.loc[i,"已签收"] = df.loc[i,'订单数']
    elif df.loc[i,"订单状态"] == "已发货" and ((df.loc[i,'商品售后'] == "无") or (df.loc[i,'商品售后'] == "用户取消申请") or (df.loc[i,'商品售后'] == "平台处理完成，售后关闭") ):
         df.loc[i,"发货中1"] = df.loc[i,'订单数']
    elif df.loc[i,'商品售后'] =="待买家退货" or df.loc[i,'商品售后'] =="待商家收货" or df.loc[i,'商品售后'] =="售后协商中" or df.loc[i,'商品售后'] =="待商家受理" :
        df.loc[i,"售后中"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"商品发货"] == "已发货" and (df.loc[i,'商品售后'] == "退货退款完成" or df.loc[i,'商品售后'] == "退款完成"):
        df.loc[i,"已退款"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"商品发货"] == "已发货" and (df.loc[i,'商品售后'] == "无" or df.loc[i,'商品售后'] == "用户取消申请" or df.loc[i,'商品售后'] == "平台处理完成，售后关闭"):
         df.loc[i,"预计签收单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已取消" and  df.loc[i,"商品发货"] == "已发货" and (df.loc[i,'商品售后'] == "用户取消申请" or df.loc[i,'商品售后'] == "平台处理完成，售后关闭" ) :
        df.loc[i,"发货中2"] = df.loc[i,'订单数']

In [177]:
import numpy as np
df['销售金额'].fillna(0, inplace=True)

df['运费险预计投保费用'] = pd.to_numeric(df['运费险预计投保费用'], errors='coerce')

result = pd.pivot_table(df,index=['下单时间',"产品名称"] , values = ['下单单量','支付单量','发货单量','已签收','发货中1','发货中2','售后中','已退款','发货前退款单量',"预计签收单量",'运费险预计投保费用','支付金额'] , aggfunc = np.sum)
result = result.reset_index()



result['运费险预计投保费用'] = np.where(result['运费险预计投保费用'].isna() | (result['运费险预计投保费用'] == 0), '关闭', '开')


result = result[['下单时间',"产品名称",'下单单量','支付单量','发货单量','已签收','发货中1','发货中2','售后中','已退款','发货前退款单量',"预计签收单量",'运费险预计投保费用','支付金额']  ]

In [178]:
#拼接店铺消耗表
df_consume = pd.read_excel("店铺消耗微信豆3号.xlsx")
df_consume = df_consume[['日期','产品名称','累计消耗金额','成交订单数','成交金额']]
temp = df_consume.日期.str.extract('(\d+).*?(\d+).*?(\d+)') 
df_consume['下单时间'] = temp[0] + "-" + temp[1] + '-' + temp[2]
del df_consume['日期']
df_consume = pd.pivot_table(df_consume,index=['下单时间','产品名称'] , values = ['累计消耗金额','成交订单数','成交金额'] , aggfunc = np.sum)
df_consume = df_consume.reset_index()
df_consume["下单时间"] = pd.to_datetime(df_consume["下单时间"],format = "%Y-%m-%d")
df_consume["下单时间"] = df_consume["下单时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df_consume

,下单时间,产品名称,成交订单数,成交金额,累计消耗金额
0,2026-01-03,2588#刺绣收腹提臀裤,0.0,0.00,27.01
1,2026-01-04,2588#刺绣收腹提臀裤,0.0,0.00,5.53
2,2026-01-05,2588#刺绣收腹提臀裤,1.0,59.90,88.64
3,2026-01-08,2588#刺绣收腹提臀裤,1.0,59.90,12.72
4,2026-01-09,2588#刺绣收腹提臀裤,0.0,0.00,11.62
5,2026-01-10,2588#刺绣收腹提臀裤,1.0,59.90,39.20
6,2026-01-11,2588#刺绣收腹提臀裤,1.0,59.90,19.89
7,2026-01-12,2588#刺绣收腹提臀裤,27.0,1687.20,1031.31
8,2026-01-13,2588#刺绣收腹提臀裤,196.0,11987.00,8043.59
9,2026-01-14,2588#刺绣收腹提臀裤,215.0,12905.50,7561.81


In [179]:
#拼接店铺消耗表

df_consume1 = pd.read_excel("店铺消耗ADQ3号.xlsx")

# 选择需要的列并提取日期
df_consume1 = df_consume1[['日期', '产品名称', '累计消耗金额', '成交订单数','成交金额']]
df_consume1['下单时间'] = df_consume1['日期'].str.extract(r'(\d{4}\.\d{1,2}\.\d{1,2})')[0]
df_consume1['下单时间'] = pd.to_datetime(df_consume1['下单时间'], format='%Y.%m.%d').dt.strftime('%Y-%m-%d')

# 删除原日期列
df_consume1.drop(columns=['日期'], inplace=True)

# 使用pivot_table进行数据透视
df_consume1 = pd.pivot_table(
    df_consume1, 
    index=['下单时间', '产品名称'], 
    values=['累计消耗金额', '成交订单数', '成交金额'], 
    aggfunc=np.sum
).reset_index()

# 将下单时间转换为日期格式
df_consume1["下单时间"] = pd.to_datetime(df_consume1["下单时间"], format='%Y-%m-%d')

# 重命名列
df_consume1.rename(columns={'成交订单数': 'ADQ成交订单数', '累计消耗金额': 'ADQ累计消耗金额', '成交金额': 'ADQ广告gmv'}, inplace=True)


print(df_consume1)

          下单时间          产品名称  ADQ成交订单数   ADQ广告gmv  ADQ累计消耗金额
0   2026-01-01  2588#刺绣收腹提臀裤       0.0       0.00       2.43
1   2026-01-02  2588#刺绣收腹提臀裤       0.0       0.00       1.29
2   2026-01-03  2588#刺绣收腹提臀裤       0.0       0.00       4.48
3   2026-01-04  2588#刺绣收腹提臀裤       0.0       0.00       7.28
4   2026-01-05  2588#刺绣收腹提臀裤       1.0      59.90      29.23
5   2026-01-06  2588#刺绣收腹提臀裤       0.0       0.00       0.09
6   2026-01-08  2588#刺绣收腹提臀裤       0.0       0.00   10562.49
7   2026-01-09  2588#刺绣收腹提臀裤       5.0     299.50    1895.80
8   2026-01-10  2588#刺绣收腹提臀裤     113.0    6778.70    7831.28
9   2026-01-11  2588#刺绣收腹提臀裤     485.0   29185.40   27628.74
10  2026-01-12  2588#刺绣收腹提臀裤     991.0   59697.64   35441.18
11  2026-01-13  2588#刺绣收腹提臀裤     621.0   37309.70   21267.49
12  2026-01-14  2588#刺绣收腹提臀裤     308.0   18486.21   11394.29
13  2026-01-15  2588#刺绣收腹提臀裤     182.0   10906.80    5404.39
14  2026-01-16  2588#刺绣收腹提臀裤     222.0   13377.70    7105.25
15  2026-01-17  2588#刺绣收

In [180]:
final = pd.merge(result,df_consume,on = ['下单时间','产品名称'],how = 'left')


final = final[['下单时间','产品名称','成交订单数','下单单量','支付单量','发货单量','已签收','发货中1','发货中2','售后中','已退款','发货前退款单量',"预计签收单量","累计消耗金额",'运费险预计投保费用','支付金额','成交金额']]
final['发货中'] = final['发货中1'] + final['发货中2']
final = final[['下单时间','产品名称','成交订单数','下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","累计消耗金额",'运费险预计投保费用','支付金额','成交金额']]
final.rename(columns={"成交订单数":"广告单量"},inplace=True)

final.rename(columns={'累计消耗金额': '微信豆累计消耗金额', '广告单量': '微信豆广告单量','成交金额': '微信豆广告gmv','运费险预计投保费用': '运费险'}, inplace=True)

df_consume1['下单时间'] = df_consume1['下单时间'].astype(str)
final = pd.merge(final, df_consume1, on=['下单时间', '产品名称'], how='left')
final['全店支付gmv'] = final['支付金额']


final = final[['下单时间','运费险','产品名称','ADQ成交订单数','微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额']]

# 消耗为空的赋值0
final.fillna(0, inplace=True)



In [181]:
def concat_unique_strings(x):
    # 步骤1：移除NaN值
    cleaned = x.dropna()
    if cleaned.empty:
        return np.nan
    
    # 步骤2：使用集合去重（不保留顺序）
    # return ','.join(set(cleaned))
    
    # 步骤3：去重且保留首次出现顺序
    seen = set()
    unique_list = [item for item in cleaned if not (item in seen or seen.add(item))]
    return ','.join(unique_list)

# 创建智能聚合字典
agg_dict = {}
for column in final.columns:
    if column == '下单时间':  # 跳过下单时间列
        continue
        
    if pd.api.types.is_numeric_dtype(final[column]):
        agg_dict[column] = 'sum'  # 数值列求和
    else:
        agg_dict[column] = concat_unique_strings  # 字符串列去重拼接


# 执行分组聚合
final_sum = final.groupby('下单时间', as_index=False).agg(agg_dict)


final['平均客单'] = final['支付金额'] / final['支付单量']
final_sum['平均客单'] = final_sum['支付金额'] / final_sum['支付单量']
final = final[['下单时间','运费险','产品名称','ADQ成交订单数','微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额','平均客单']]
final_sum = final_sum[['下单时间','运费险','产品名称','ADQ成交订单数','微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额','平均客单']]


In [182]:
final_sum

,下单时间,运费险,产品名称,ADQ成交订单数,微信豆广告单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,ADQ累计消耗金额,微信豆累计消耗金额,全店支付gmv,微信豆广告gmv,ADQ广告gmv,支付金额,平均客单
0,2026-10-17,关闭,2588#刺绣收腹提臀裤,0.0,26.0,27,24,21,1,0,2,3,22,0.0,506.58,1617.3,1557.40,0.0,1617.3,59.900000
1,2026-10-18,关闭,2588#刺绣收腹提臀裤,0.0,26.0,36,33,27,0,0,6,3,27,0.0,887.04,2176.4,2173.40,0.0,2176.4,60.455556
2,2026-10-19,关闭,2588#刺绣收腹提臀裤,0.0,19.0,19,18,16,0,0,2,1,16,0.0,675.83,1138.1,1135.10,0.0,1138.1,59.900000
3,2026-10-20,关闭,2588#刺绣收腹提臀裤,0.0,125.0,129,121,91,1,0,29,8,92,0.0,4398.69,7757.1,7687.20,0.0,7757.1,60.132558
4,2026-10-21,关闭,2588#刺绣收腹提臀裤,0.0,50.0,54,50,37,0,0,13,4,37,0.0,1904.47,3234.6,2964.00,0.0,3234.6,59.900000
5,2026-10-22,关闭,2588#刺绣收腹提臀裤,0.0,57.0,61,56,46,0,0,10,5,46,0.0,1813.52,3663.9,3464.20,0.0,3663.9,60.063934
6,2026-10-23,关闭,2588#刺绣收腹提臀裤,0.0,53.0,72,69,52,1,1,15,3,53,0.0,1803.00,4312.8,3231.60,0.0,4312.8,59.900000
7,2026-10-24,关闭,2588#刺绣收腹提臀裤,0.0,152.0,189,178,150,1,1,26,11,151,0.0,4186.41,11351.1,9179.70,0.0,11351.1,60.058730
8,2026-10-25,关闭,2588#刺绣收腹提臀裤,0.0,397.0,420,400,351,0,5,44,20,351,0.0,11130.84,25178.0,23952.18,0.0,25178.0,59.947619
9,2026-10-26,关闭,2588#刺绣收腹提臀裤,0.0,144.0,158,149,111,11,5,22,9,122,0.0,4426.22,9494.2,8707.50,0.0,9494.2,60.089873


In [183]:
# writer = pd.ExcelWriter('直播汇总.xlsx')
# final.to_excel(writer, sheet_name='汇总', index=False)
# writer.save()
writer = pd.ExcelWriter('直播汇总.xlsx')
final_sum.to_excel(writer, sheet_name='汇总', index=False)
final.to_excel(writer, sheet_name='产品分类', index=False)
writer.close()  # 改用 close() 方法

In [184]:
df = pd.read_excel('直播汇总.xlsx', sheet_name='汇总')  # 读取 Excel 文件中的汇总表例如，从 CSV 文件加载数据
len(df)

20

In [185]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string

def write_data_to_range(ws, data, 开始行, 开始列, 结束行, 结束列):
    """
    将数据写入Excel工作表指定范围
    
    参数:
        ws: Excel工作表对象
        data: 二维数据列表
        开始行: 起始行号(整数)
        开始列: 起始列字母(如'A', 'B')
        结束行: 结束行号(整数)
        结束列: 结束列字母(如'C', 'D')
    """
    # 将字母列转换为数字索引
    start_col_num = column_index_from_string(开始列)
    end_col_num = column_index_from_string(结束列)
    
    # 遍历指定范围
    for row_idx in range(开始行, 结束行 + 1):
        for col_idx in range(start_col_num, end_col_num + 1):
            # 计算数据中的位置
            data_row = row_idx - 开始行
            data_col = col_idx - start_col_num
            
            # 检查数据索引是否有效
            if data_row < len(data) and data_col < len(data[data_row]):
                value = data[data_row][data_col]
                ws.cell(row=row_idx, column=col_idx, value=value)

df = pd.read_excel('直播汇总.xlsx', sheet_name='汇总')  # 读取 Excel 文件中的汇总表例如，从 CSV 文件加载数据

product_name = df.loc[0, '产品名称']

# 指定要写入的列和前20行数据
columns_to_write = ['下单时间', '运费险', 'ADQ成交订单数', '微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额']
if len(df) > 20:
    data_to_write = df[columns_to_write].head(20)
    df_length = 20
else:
    df_length = len(df)
    data_to_write = df[columns_to_write]

# Excel 文件路径
excel_file = './视频号_签收记录表自动化测试.xlsx'

try:
    wb = load_workbook(excel_file)
    
    # 确保至少有一个可见的工作表
    if all(sheet.sheet_state == 'hidden' for sheet in wb.worksheets):
        wb.worksheets[0].sheet_state = 'visible'
    
    for sheet in wb.worksheets:
        if sheet.title != "原始表格":
            del wb[sheet.title]
    
    # 复制或创建输出表格
    sheet_name = '原始表格'
    
    if sheet_name in wb.sheetnames:
        # 获取原始表格
        original_ws = wb[sheet_name]
        
        # 创建输出表格（复制原始表格）
        output_ws = wb.copy_worksheet(original_ws)
        output_ws.title = "输出表格"
        
        # 将输出表格移动到第一个位置
        sheets = wb.worksheets
        output_index = sheets.index(output_ws)
        if output_index > 0:
            sheets.pop(output_index)
            sheets.insert(0, output_ws)
            wb._sheets = sheets
        
        # 现在使用输出表格
        ws = output_ws
    else:
        # 创建新的输出表格
        ws = wb.create_sheet("输出表格")
        
        # 移动到第一个位置
        wb.move_sheet(ws, offset=-len(wb.sheetnames))
    
    # 准备数据（转换为列表格式）
    data = data_to_write.values.tolist()
    print(data)

    # 写入数据到 B4-E23 范围 (4列)
    write_data_to_range(
        ws,
        [row[:4] for row in data],
        开始行=4,
        开始列="B",
        结束行=23,
        结束列="E"
    )
    # 写入数据到 E4-P23 范围 (8列)
    write_data_to_range(
        ws,
        [row[4:12] for row in data],
        开始行=4,
        开始列="I",
        结束行=23,
        结束列="P"
    )
    # 写入数据到 W4-AA23 范围 (2列)
    write_data_to_range(
        ws,
        [row[12:17] for row in data],
        开始行=4,
        开始列="W",
        结束行=23,
        结束列="AA"
    )
    write_data_to_range(
        ws,
        [row[17:18] for row in data],
        开始行=4,
        开始列="AE",
        结束行=23,
        结束列="AF"
    )
    # 保存工作簿
    wb.save(excel_file)
    print("数据写入成功！")
    
except PermissionError:
    print("错误：文件被其他程序锁定，请关闭后重试")
except FileNotFoundError:
    print(f"错误：文件 {excel_file} 不存在")
except Exception as e:
    print(f"处理文件时出错: {e}")


[['2026-10-17', '关闭', 0, 26, 27, 24, 21, 1, 0, 2, 3, 22, 0, 506.58, 1617.300000000001, 1557.4, 0, 1617.300000000001], ['2026-10-18', '关闭', 0, 26, 36, 33, 27, 0, 0, 6, 3, 27, 0, 887.04, 2176.400000000001, 2173.4, 0, 2176.400000000001], ['2026-10-19', '关闭', 0, 19, 19, 18, 16, 0, 0, 2, 1, 16, 0, 675.83, 1138.1, 1135.1, 0, 1138.1], ['2026-10-20', '关闭', 0, 125, 129, 121, 91, 1, 0, 29, 8, 92, 0, 4398.69, 7757.099999999984, 7687.2, 0, 7757.099999999984], ['2026-10-21', '关闭', 0, 50, 54, 50, 37, 0, 0, 13, 4, 37, 0, 1904.47, 3234.600000000003, 2964.0, 0, 3234.600000000003], ['2026-10-22', '关闭', 0, 57, 61, 56, 46, 0, 0, 10, 5, 46, 0, 1813.52, 3663.900000000004, 3464.2, 0, 3663.900000000004], ['2026-10-23', '关闭', 0, 53, 72, 69, 52, 1, 1, 15, 3, 53, 0, 1803.0, 4312.800000000003, 3231.6, 0, 4312.800000000003], ['2026-10-24', '关闭', 0, 152, 189, 178, 150, 1, 1, 26, 11, 151, 0, 4186.41, 11351.09999999996, 9179.7, 0, 11351.09999999996], ['2026-10-25', '关闭', 0, 397, 420, 400, 351, 0, 5, 44, 20, 351, 0, 1

In [186]:
data

[['2026-10-17',
  '关闭',
  0,
  26,
  27,
  24,
  21,
  1,
  0,
  2,
  3,
  22,
  0,
  506.58,
  1617.300000000001,
  1557.4,
  0,
  1617.300000000001],
 ['2026-10-18',
  '关闭',
  0,
  26,
  36,
  33,
  27,
  0,
  0,
  6,
  3,
  27,
  0,
  887.04,
  2176.400000000001,
  2173.4,
  0,
  2176.400000000001],
 ['2026-10-19',
  '关闭',
  0,
  19,
  19,
  18,
  16,
  0,
  0,
  2,
  1,
  16,
  0,
  675.83,
  1138.1,
  1135.1,
  0,
  1138.1],
 ['2026-10-20',
  '关闭',
  0,
  125,
  129,
  121,
  91,
  1,
  0,
  29,
  8,
  92,
  0,
  4398.69,
  7757.099999999984,
  7687.2,
  0,
  7757.099999999984],
 ['2026-10-21',
  '关闭',
  0,
  50,
  54,
  50,
  37,
  0,
  0,
  13,
  4,
  37,
  0,
  1904.47,
  3234.600000000003,
  2964.0,
  0,
  3234.600000000003],
 ['2026-10-22',
  '关闭',
  0,
  57,
  61,
  56,
  46,
  0,
  0,
  10,
  5,
  46,
  0,
  1813.52,
  3663.900000000004,
  3464.2,
  0,
  3663.900000000004],
 ['2026-10-23',
  '关闭',
  0,
  53,
  72,
  69,
  52,
  1,
  1,
  15,
  3,
  53,
  0,
  1803.0,
  4312